# Block 1: Regression for Household Demand Forecasting
## Hochfrequenz AI Workshop

**Objective:** Build baseline and ML models to predict next-hour household electricity consumption

**Data:** German smart meter dataset (38 households, 2018-2020, Nature Scientific Data)

**Models:** Baseline (seasonal) → Linear Regression → Random Forest → XGBoost

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for nice plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ All libraries imported successfully")

### 1.1 Load & Explore Data

For this workshop, we'll generate realistic synthetic data matching the German smart meter distribution. In production, you'd load real data from Zenodo (link below).

In [ ]:
# Generate realistic synthetic German household consumption data
# This mimics the Nature Scientific Data dataset structure

np.random.seed(42)

# Generate 2 years of hourly data
date_range = pd.date_range(start='2018-05-01', end='2020-12-31', freq='H')
n_samples = len(date_range)

# Create realistic consumption pattern
data = pd.DataFrame({
    'timestamp': date_range,
    'hour': date_range.hour,
    'day_of_week': date_range.dayofweek,
    'month': date_range.month,
    'is_weekend': (date_range.dayofweek >= 5).astype(int)
})

# Base consumption pattern (W) - typical German household
# Morning peak (~7 AM), evening peak (~6-7 PM), low at night
hourly_pattern = np.array([
    300, 280, 250, 240, 250, 350, 550, 700,  # Night to morning
    600, 500, 450, 420, 430, 480, 550, 600,  # Midday
    700, 800, 750, 700, 650, 600, 550, 400   # Evening to night
])

# Apply hourly pattern
data['consumption_base'] = hourly_pattern[data['hour']]

# Add seasonal variation (winter = higher consumption)
seasonal_factor = 1.0 + 0.3 * np.sin(2 * np.pi * (data['month'] - 2) / 12)
data['consumption_seasonal'] = data['consumption_base'] * seasonal_factor

# Add weekend effect (lower consumption)
data['consumption_adjusted'] = data['consumption_seasonal'] * (0.85 if data['is_weekend'] else 1.0)

# Add random noise and weather effects
data['consumption'] = data['consumption_adjusted'] + np.random.normal(0, 50, n_samples)
data['consumption'] = data['consumption'].clip(lower=50)  # No negative consumption

# Create a simple temperature proxy (correlates with season)
data['temperature'] = 15 + 8 * np.sin(2 * np.pi * (data['month'] - 2) / 12) + np.random.normal(0, 2, n_samples)

# Set timestamp as index
data.set_index('timestamp', inplace=True)

print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
print(f"\nFirst 5 rows:")
print(data.head())
print(f"\nConsumption statistics:")
print(data['consumption'].describe())

### 1.2 Visualize Consumption Patterns (Where's EDA?)

In [ ]:
# Plot: Daily and seasonal patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Hourly pattern (average across all days)
hourly_avg = data.groupby('hour')['consumption'].mean()
axes[0, 0].plot(hourly_avg.index, hourly_avg.values, 'o-', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Avg Consumption (W)')
axes[0, 0].set_title('Typical Daily Consumption Pattern')
axes[0, 0].grid(True, alpha=0.3)

# 2. Seasonal pattern (average by month)
monthly_avg = data.groupby('month')['consumption'].mean()
axes[0, 1].plot(monthly_avg.index, monthly_avg.values, 'o-', linewidth=2, markersize=6, color='orange')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Avg Consumption (W)')
axes[0, 1].set_title('Seasonal Consumption Pattern')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].grid(True, alpha=0.3)

# 3. Time series - first 30 days
data_subset = data['consumption'].iloc[:30*24]
axes[1, 0].plot(data_subset.index, data_subset.values, linewidth=1, alpha=0.7)
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Consumption (W)')
axes[1, 0].set_title('First 30 Days - Time Series')
axes[1, 0].grid(True, alpha=0.3)

# 4. Weekend vs Weekday
weekday_avg = data[data['is_weekend']==0].groupby('hour')['consumption'].mean()
weekend_avg = data[data['is_weekend']==1].groupby('hour')['consumption'].mean()
axes[1, 1].plot(weekday_avg.index, weekday_avg.values, 'o-', label='Weekday', linewidth=2)
axes[1, 1].plot(weekend_avg.index, weekend_avg.values, 's-', label='Weekend', linewidth=2)
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Avg Consumption (W)')
axes[1, 1].set_title('Weekday vs Weekend Pattern')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Data patterns visualized")

## 2. Feature Engineering

Create features that help models learn consumption patterns

In [ ]:
# Prepare target: next-hour consumption
data['consumption_next'] = data['consumption'].shift(-1)

# Create lagged features (historical consumption)
for lag in [1, 2, 3, 24, 48]:  # 1h, 2h, 3h, 1 day, 2 days ago
    data[f'consumption_lag_{lag}'] = data['consumption'].shift(lag)

# Create rolling statistics
data['consumption_rolling_mean_24'] = data['consumption'].rolling(window=24).mean()
data['consumption_rolling_std_24'] = data['consumption'].rolling(window=24).std()

# Cyclical encoding for hour (sin/cos to capture circularity)
data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

# Cyclical encoding for month
data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)

# Drop rows with NaN (from lag and rolling features)
data_clean = data.dropna()

print(f"Features created. Dataset shape after removing NaNs: {data_clean.shape}")
print(f"\nFeature columns:")
feature_cols = [col for col in data_clean.columns if col != 'consumption_next']
print(feature_cols)
print(f"\nTarget: consumption_next")

## 3. Train-Test Split & Baseline Model

**Time series rule:** Never shuffle! Use chronological split (train on past, test on future)

In [ ]:
# Chronological split: 80% train, 20% test
split_idx = int(len(data_clean) * 0.8)
data_train = data_clean.iloc[:split_idx]
data_test = data_clean.iloc[split_idx:]

print(f"Train set: {data_train.index.min()} to {data_train.index.max()} ({len(data_train)} samples)")
print(f"Test set: {data_test.index.min()} to {data_test.index.max()} ({len(data_test)} samples)")

# Prepare X and y
X_train = data_train.drop('consumption_next', axis=1)
y_train = data_train['consumption_next']

X_test = data_test.drop('consumption_next', axis=1)
y_test = data_test['consumption_next']

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

### 3.1 Baseline Model: Seasonal Average

In [ ]:
# Baseline: Use historical average for same hour + day-of-week
baseline_pred = []

for idx, row in data_test.iterrows():
    hour = row['hour']
    dow = row['day_of_week']
    
    # Find average consumption for this hour & day-of-week in training set
    mask = (data_train['hour'] == hour) & (data_train['day_of_week'] == dow)
    avg = data_train[mask]['consumption_next'].mean()
    baseline_pred.append(avg)

baseline_pred = np.array(baseline_pred)

# Evaluate baseline
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline Model (Seasonal Average) Performance:")
print(f"  MAE:  {baseline_mae:.2f} W")
print(f"  RMSE: {baseline_rmse:.2f} W")
print(f"  R²:   {baseline_r2:.3f}")
print(f"\n  Interpretation:")
print(f"  - On average, prediction is off by {baseline_mae:.0f} W")
print(f"  - Model explains {baseline_r2*100:.1f}% of variance")

## 4. Train ML Models

### 4.1 Linear Regression

In [ ]:
# Scale features for linear regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict
lr_pred_train = lr_model.predict(X_train_scaled)
lr_pred_test = lr_model.predict(X_test_scaled)

# Evaluate
lr_mae = mean_absolute_error(y_test, lr_pred_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred_test))
lr_r2 = r2_score(y_test, lr_pred_test)

print("Linear Regression Performance:")
print(f"  MAE:  {lr_mae:.2f} W")
print(f"  RMSE: {lr_rmse:.2f} W")
print(f"  R²:   {lr_r2:.3f}")
print(f"\n  vs Baseline:")
print(f"  MAE improvement: {(baseline_mae - lr_mae) / baseline_mae * 100:.1f}%")

### 4.2 Random Forest

In [ ]:
# Train Random Forest (no scaling needed for tree-based models)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict
rf_pred_train = rf_model.predict(X_train)
rf_pred_test = rf_model.predict(X_test)

# Evaluate
rf_mae = mean_absolute_error(y_test, rf_pred_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred_test))
rf_r2 = r2_score(y_test, rf_pred_test)

print("Random Forest Performance:")
print(f"  MAE:  {rf_mae:.2f} W")
print(f"  RMSE: {rf_rmse:.2f} W")
print(f"  R²:   {rf_r2:.3f}")
print(f"\n  vs Baseline:")
print(f"  MAE improvement: {(baseline_mae - rf_mae) / baseline_mae * 100:.1f}%")

### 4.3 XGBoost (Gradient Boosting)

In [ ]:
# Train XGBoost
xgb_model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train, verbose=False)

# Predict
xgb_pred_train = xgb_model.predict(X_train)
xgb_pred_test = xgb_model.predict(X_test)

# Evaluate
xgb_mae = mean_absolute_error(y_test, xgb_pred_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred_test))
xgb_r2 = r2_score(y_test, xgb_pred_test)

print("XGBoost Performance:")
print(f"  MAE:  {xgb_mae:.2f} W")
print(f"  RMSE: {xgb_rmse:.2f} W")
print(f"  R²:   {xgb_r2:.3f}")
print(f"\n  vs Baseline:")
print(f"  MAE improvement: {(baseline_mae - xgb_mae) / baseline_mae * 100:.1f}%")

## 5. Model Comparison

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['Baseline (Seasonal)', 'Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE (W)': [baseline_mae, lr_mae, rf_mae, xgb_mae],
    'RMSE (W)': [baseline_rmse, lr_rmse, rf_rmse, xgb_rmse],
    'R²': [baseline_r2, lr_r2, rf_r2, xgb_r2]
})

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison.to_string(index=False))
print("="*60)

# Highlight winner
best_model_idx = comparison['MAE (W)'].idxmin()
print(f"\n🏆 Best Model: {comparison.iloc[best_model_idx]['Model']}")
print(f"   MAE: {comparison.iloc[best_model_idx]['MAE (W)']:.2f} W")

## 6. Visualization: Actual vs Predicted

In [ ]:
# Plot predictions for a subset of test data
test_subset = slice(0, 7*24)  # First week of test set

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
models_to_plot = [
    ('Baseline (Seasonal)', baseline_pred[test_subset]),
    ('Linear Regression', lr_pred_test[test_subset]),
    ('Random Forest', rf_pred_test[test_subset]),
    ('XGBoost', xgb_pred_test[test_subset])
]

actual = y_test.values[test_subset]
x_axis = range(len(actual))

for idx, (ax, (model_name, predictions)) in enumerate(zip(axes.flatten(), models_to_plot)):
    ax.plot(x_axis, actual, 'o-', label='Actual', linewidth=2, markersize=4, alpha=0.7)
    ax.plot(x_axis, predictions, 's--', label='Predicted', linewidth=2, markersize=4, alpha=0.7)
    ax.set_ylabel('Consumption (W)')
    ax.set_title(f'{model_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add MAE annotation
    mae = mean_absolute_error(actual, predictions)
    ax.text(0.98, 0.05, f'MAE: {mae:.1f} W', transform=ax.transAxes, 
           ha='right', va='bottom', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('First Week of Test Predictions - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# Feature importance from tree-based models
rf_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

xgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 10 features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(10), rf_importance['importance'].head(10))
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(rf_importance['feature'].head(10))
axes[0].set_xlabel('Importance')
axes[0].set_title('Random Forest - Top 10 Features')
axes[0].invert_yaxis()

axes[1].barh(range(10), xgb_importance['importance'].head(10), color='orange')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(xgb_importance['feature'].head(10))
axes[1].set_xlabel('Importance')
axes[1].set_title('XGBoost - Top 10 Features')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n📊 Top 5 Important Features (Random Forest):")
for idx, row in rf_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.3f}")

## 8. Key Takeaways

### What We Learned:

1. **Baseline Matters**: We started with a simple seasonal baseline (~{:.0f}W MAE)
2. **ML Improves Predictions**: Best model achieves ~{:.0f}W MAE ({:.1f}% improvement). 
3. **Feature Engineering is Critical**: Lagged consumption, hour/month encoding, rolling statistics all help. Instead of spending hours on randomly fitting different models, think about your data. Maybe a log transform will boost predictions.
4. **Tree-Based Models (RF, XGBoost) Beat Linear**: Non-linear relationships exist in consumption patterns
5. **Feature Importance Shows Domain Knowledge**: Hour, lagged consumption, temperature matter most

### Business Impact (For Hochfrequenz Clients):

- **Billing Accuracy**: Better forecasts → more accurate estimated bills → fewer disputes
- **Grid Operations**: Accurate forecasts → better demand planning → cost savings
- **Customer Programs**: Segment customers by patterns (high/low demand) → targeted efficiency programs

---

### Next: Block 2 - LSTM for Multi-Step Forecasting
Deep learning will let us predict not just 1 hour ahead, but 24 hours ahead!
""".format(int(baseline_mae), int(xgb_mae), (baseline_mae - xgb_mae) / baseline_mae * 100)